In [ ]:
%pip install requests #pandas #if needed

In [ ]:
from time import monotonic
import time, json
from pathlib import Path

import requests
import pandas as pd

BASE_URL = "https://oqmd.org/oqmdapi/formationenergy"

PAGE_SIZE = 50 # not important for small queeries, but if filtering for large datasets may want to change. Smaller pages are typically more reliable


#List desired structures here. 
#----------------------------------------------------------------------
STRUCTURES = [
    "NaMnO2",
    "LiFePO4",
    "LiCoO2",
    "NaFeO2",
    "SrTiO3"
]
#----------------------------------------------------------------------

# Optional extra filter, e.g.:
# EXTRA_FILTER = "stability<0.1 AND ntypes>=2"

EXTRA_FILTER = None



#----------------------------------------------------------------------
OUT_DIR = Path("/Users/benhopkins/Documents/GitHub/OQMD/oqmd_downloads") # Change this to your desired output directory 
OUT_DIR.mkdir(parents=True, exist_ok=True)                         # it will save checkpoints and final results here.
#----------------------------------------------------------------------


#Checkpoints for larger downloads - saves progress after each page of results, so you can resume if interrupted.
CHECKPOINT_CSV = OUT_DIR / "oqmd_structure_search_checkpoint.csv"
CHECKPOINT_JSON = OUT_DIR / "oqmd_structure_search_checkpoint.json"
OUTPUT_CSV = OUT_DIR / "oqmd_structure_search_clean.csv"

#Data each search returns - you can add/remove fields as needed, but make sure to update the normalise_record function accordingly.
FIELDS = [
    "name",
    "entry_id",
    "formationenergy_id",
    "duplicate_entry_id",
    "icsd_id",
    "prototype",
    "spacegroup",
    "composition",
    "composition_generic",
    "ntypes",
    "natoms",
    "volume",
    "unit_cell",
    "sites",
    "delta_e",
    "stability",
    "band_gap",
    "fit",
    "calculation_label",
]

PARAMS_BASE = {
    "fields": ",".join(FIELDS),
    "limit": PAGE_SIZE,
    "format": "json",
    "noduplicate": "False",
}

# Function to handle GET requests with retries and exponential backoff

def get_with_retries(url, params, max_retries=8):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=(10, 300))
            if r.status_code == 200:
                return r
            print(f"Attempt {attempt + 1}: HTTP {r.status_code}")
            print(r.text[:500])
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1}: {type(e).__name__} - {e}")

        sleep_time = 5 * (attempt + 1)
        print(f"Retrying in {sleep_time} seconds...")
        time.sleep(sleep_time)

    raise RuntimeError("Request failed after all retry attempts")

# Normalise the record to a consistent format, and convert complex fields to JSON strings
def normalise_record(row):
    return {
        "queried_structure": row.get("name"),
        "name": row.get("name"),
        "entry_id": row.get("entry_id"),
        "formationenergy_id": row.get("formationenergy_id"),
        "duplicate_entry_id": row.get("duplicate_entry_id"),
        "icsd_id": row.get("icsd_id"),
        "prototype": row.get("prototype"),
        "spacegroup": row.get("spacegroup"),
        "composition": row.get("composition"),
        "composition_generic": row.get("composition_generic"),
        "ntypes": row.get("ntypes"),
        "natoms": row.get("natoms"),
        "volume_A3": row.get("volume"),
        "formation_energy_eV_per_atom": row.get("delta_e"),
        "stability_eV_per_atom": row.get("stability"),
        "band_gap_eV": row.get("band_gap"),
        "fit": row.get("fit"),
        "calculation_label": row.get("calculation_label"),
        "unit_cell_json": json.dumps(row.get("unit_cell"), ensure_ascii=False),
        "sites_json": json.dumps(row.get("sites"), ensure_ascii=False),
    }

# Main loop to query each structure, handle pagination, and save results
# Can basically ignore everything below this point - it's just handling the API querying, pagination, and saving results to CSV/JSON checkpoints and final output.
records = []
start = monotonic()

for formula in STRUCTURES:
    offset = 0
    page_number = 0
    total_entries = None

    print(f"\nSearching for {formula}...")

    while True:
        params = dict(PARAMS_BASE)
        params["composition"] = formula
        params["offset"] = offset

        if EXTRA_FILTER:
            params["filter"] = EXTRA_FILTER

        response = get_with_retries(BASE_URL, params)

        try:
            payload = response.json()
        except ValueError:
            print("Failed to decode JSON.")
            print(response.text[:1000])
            break

        meta = payload.get("meta", {})
        data = payload.get("data", [])

        if total_entries is None:
            total_entries = meta.get("data_available", len(data))
            print(f"{formula}: total entries available: {total_entries}")

        if not data:
            print(f"No more data returned for {formula}.")
            break

        page_number += 1

        for row in data:
            record = normalise_record(row)
            record["queried_structure"] = formula
            records.append(record)

        df_checkpoint = pd.DataFrame(records)
        df_checkpoint.to_csv(CHECKPOINT_CSV, index=False)

        offset += len(data)

        CHECKPOINT_JSON.write_text(json.dumps({
            "current_formula": formula,
            "offset": offset,
            "page_number": page_number,
            "total_entries_current_formula": total_entries,
            "records_downloaded_total": len(records),
            "structures": STRUCTURES,
            "extra_filter": EXTRA_FILTER,
            "fields": FIELDS,
        }, indent=2))

        print(
            f"{formula} page {page_number} complete | "
            f"Downloaded for this formula: {offset} / {total_entries} | "
            f"Total records: {len(records)}"
        )

        if not meta.get("more_data_available", False):
            break

end = monotonic()

df = pd.DataFrame(records)

numeric_cols = [
    "formation_energy_eV_per_atom",
    "stability_eV_per_atom",
    "band_gap_eV",
    "volume_A3",
    "natoms",
    "ntypes",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["formation_energy_eV_per_atom"])
df = df.drop_duplicates(subset=["entry_id", "formationenergy_id"])
df = df.reset_index(drop=True)

df.to_csv(OUTPUT_CSV, index=False)

print(f"\nQuery took {end - start:.2f} seconds")
print(f"Final cleaned records: {len(df)}")
print(f"Saved to: {OUTPUT_CSV}")
print(df.head())


Searching for NaMnO2...
Attempt 1: HTTP 502

<html><head>
<meta http-equiv="content-type" content="text/html;charset=utf-8">
<title>502 Server Error</title>
</head>
<body text=#000000 bgcolor=#ffffff>
<h1>Error: Server Error</h1>
<h2>The server encountered a temporary error and could not complete your request.<p>Please try again in 30 seconds.</h2>
<h2></h2>
</body></html>

Retrying in 5 seconds...
NaMnO2: total entries available: 14
NaMnO2 page 1 complete | Downloaded for this formula: 14 / 14 | Total records: 14

Searching for LiFePO4...
LiFePO4: total entries available: 4
LiFePO4 page 1 complete | Downloaded for this formula: 4 / 4 | Total records: 18

Searching for LiCoO2...
LiCoO2: total entries available: 26
LiCoO2 page 1 complete | Downloaded for this formula: 26 / 26 | Total records: 44

Searching for NaFeO2...
NaFeO2: total entries available: 16
NaFeO2 page 1 complete | Downloaded for this formula: 16 / 16 | Total records: 60

Searching for SrTiO3...
SrTiO3: total entries ava